# QUANT — Class Weight and Probability Threshold Search

This notebook extends the working five-feature QUANT notebook to tune both:

- the `ExtraTreesClassifier` **class weight**, and
- the positive-class **probability threshold**.

It uses only:

1. `TOTUSJH`
2. `TOTBSQ`
3. `TOTPOT`
4. `TOTUSJZ`
5. `ABSNJZH`

The standardized combined training tensor is expected to have shape approximately
`(277010, 24, 60)`, with the channel order defined by
`feature_columns_final_magnetic.json`.

## Validation design

The notebook keeps the supplied working notebook's structure:

```text
X_train/y_train
    -> stratified model-train/validation split
    -> fit QUANT transform once
    -> search class weights and thresholds on validation data
    -> refit the winning Extra Trees model on all transformed training data
    -> evaluate once on X_test/y_test (Partition 4)
```

No partition-ID reconstruction or manual partition counts are required. The held-out
`X_test`/`y_test` data are not used to select the class weight or threshold.


## 1. Mount Google Drive and install packages


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import importlib.metadata
import subprocess
import sys

REQUIRED_AEON_VERSION = "1.5.0"

try:
    installed_aeon = importlib.metadata.version("aeon")
except importlib.metadata.PackageNotFoundError:
    installed_aeon = None

if installed_aeon != REQUIRED_AEON_VERSION:
    print(f"Installing aeon=={REQUIRED_AEON_VERSION} (currently installed: {installed_aeon})")
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"aeon=={REQUIRED_AEON_VERSION}",
        "scikit-learn",
        "joblib",
        "pandas",
        "tqdm",
    ])
else:
    print(f"aeon=={REQUIRED_AEON_VERSION} is already installed.")


## 2. Imports and configuration


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import time

import aeon
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from tqdm.auto import tqdm

from aeon.transformations.collection.interval_based import QUANTTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

print("aeon version:", aeon.__version__)
print("scikit-learn version:", sklearn.__version__)


In [ ]:
# =============================
# Data and output paths
# =============================
DATA_DIR = Path(
    "/content/drive/MyDrive/solar_flare_forecasting/Data/model_ready_partition_split/final_clean_magnetic_only"
)
OUTPUT_DIR = DATA_DIR / "quant_5_feature_class_weight_threshold_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =============================
# Features
# =============================
FEATURES_TO_USE = [
    "TOTUSJH",
    "TOTBSQ",
    "TOTPOT",
    "TOTUSJZ",
    "ABSNJZH",
]

# =============================
# Optional smoke-test sampling
# Keep both as None for the full experiment.
# =============================
TRAIN_SAMPLE_SIZE = None       # Example: 20_000
TEST_SAMPLE_SIZE = None        # Example: 5_000
RANDOM_SEED = 42

# =============================
# Validation and search grid
# =============================
VALIDATION_SIZE = 0.20
CLASS_WEIGHT_OPTIONS = [
    ("none", None),
    ("balanced", "balanced"),
    ("positive_5x", {0: 1, 1: 5}),
    ("positive_10x", {0: 1, 1: 10}),
]
THRESHOLDS = np.linspace(0.05, 0.95, 19)
DEFAULT_THRESHOLD = 0.50

# =============================
# QUANT and Extra Trees settings
# =============================
QUANT_INTERVAL_DEPTH = 6
QUANT_QUANTILE_DIVISOR = 4
N_ESTIMATORS = 200
PREDICT_BATCH_SIZE = 20_000

# =============================
# Input validity and caching
# =============================
DROP_LOW_VARIANCE_CASES = True
VARIANCE_THRESHOLD = 1e-7
CACHE_TRANSFORMED_FEATURES = True
REUSE_VALID_CACHE = True
SAVE_FINAL_MODEL = True

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("Features:", FEATURES_TO_USE)
print("Class weights:", [name for name, _ in CLASS_WEIGHT_OPTIONS])
print("Thresholds:", THRESHOLDS)


## 3. Locate and load the standardized tensors


In [ ]:
def find_first_existing(data_dir: Path, candidates: list[str]) -> Path:
    for name in candidates:
        path = data_dir / name
        if path.exists():
            return path
    checked = "\n".join(str(data_dir / name) for name in candidates)
    raise FileNotFoundError(f"None of the expected files were found. Checked:\n{checked}")


x_train_path = find_first_existing(DATA_DIR, [
    "X_train.npy"
])
x_test_path = find_first_existing(DATA_DIR, [
    "X_test.npy"
])
y_train_path = find_first_existing(DATA_DIR, ["y_train.npy"])
y_test_path = find_first_existing(DATA_DIR, ["y_test.npy"])
feature_columns_path = find_first_existing(DATA_DIR, [
    "feature_columns_final_magnetic.json"
])

print("X train:", x_train_path)
print("X test: ", x_test_path)
print("y train:", y_train_path)
print("y test: ", y_test_path)
print("Feature columns:", feature_columns_path)


In [ ]:
# Memory-map the large tensors. Only the five selected channels are copied later.
X_train_raw = np.load(x_train_path, mmap_mode="r")
X_test_raw = np.load(x_test_path, mmap_mode="r")
y_train_raw = np.load(y_train_path)
y_test_raw = np.load(y_test_path)

with open(feature_columns_path, "r") as f:
    feature_columns = json.load(f)

print("Raw shapes and dtypes")
print("X_train:", X_train_raw.shape, X_train_raw.dtype)
print("X_test: ", X_test_raw.shape, X_test_raw.dtype)
print("y_train:", y_train_raw.shape, y_train_raw.dtype)
print("y_test: ", y_test_raw.shape, y_test_raw.dtype)
print("Feature count:", len(feature_columns))


## 4. Select the five requested channels


In [ ]:
missing_features = [f for f in FEATURES_TO_USE if f not in feature_columns]
if missing_features:
    raise ValueError(f"Requested features missing from the feature-column file: {missing_features}")

selected_indices = [feature_columns.index(f) for f in FEATURES_TO_USE]
feature_index_map = dict(zip(FEATURES_TO_USE, selected_indices))

print("Selected feature indices")
for feature, index in feature_index_map.items():
    print(f"  {feature:8s} -> {index}")


def select_channels_as_aeon(X, indices, total_feature_count):
    """Return five channels in aeon layout: cases x channels x timepoints."""
    if X.ndim != 3:
        raise ValueError(f"Expected a 3D tensor, received {X.shape}")

    if X.shape[1] == total_feature_count:
        selected = X[:, indices, :]
        layout = "cases x channels x timepoints"
    elif X.shape[2] == total_feature_count:
        selected = X[:, :, indices].transpose(0, 2, 1)
        layout = "cases x timepoints x channels -> transposed"
    else:
        raise ValueError(
            f"Could not identify the feature axis. Tensor shape={X.shape}, "
            f"feature-column count={total_feature_count}."
        )

    return np.asarray(selected, dtype=np.float32), layout


X_train_5, train_layout = select_channels_as_aeon(
    X_train_raw, selected_indices, len(feature_columns)
)
X_test_5, test_layout = select_channels_as_aeon(
    X_test_raw, selected_indices, len(feature_columns)
)

print("Train layout:", train_layout)
print("Test layout: ", test_layout)
print("Selected train shape:", X_train_5.shape)
print("Selected test shape: ", X_test_5.shape)

# Close the original memory maps after the five channels have been copied.
del X_train_raw, X_test_raw
gc.collect()


## 5. Validate, optionally sample, and remove invalid low-variance cases


In [ ]:
def class_count_table(y):
    values, counts = np.unique(y, return_counts=True)
    table = pd.DataFrame({"class": values, "count": counts})
    table["percent"] = 100 * table["count"] / len(y)
    return table


def stratified_sample(X, y, sample_size, random_seed):
    original_indices = np.arange(len(y))
    if sample_size is None or sample_size >= len(y):
        return X, y, original_indices

    selected, _ = train_test_split(
        original_indices,
        train_size=sample_size,
        stratify=y,
        random_state=random_seed,
    )
    selected = np.sort(selected)
    return X[selected], y[selected], selected


def remove_low_variance_cases(X, y, original_indices, threshold, name):
    per_case_channel_std = np.std(X, axis=2)
    keep_mask = (per_case_channel_std > threshold).all(axis=1)
    removed = int((~keep_mask).sum())
    print(f"{name}: removing {removed:,} / {len(y):,} cases with a low-variance selected channel")

    if removed:
        bad_pairs = np.argwhere(per_case_channel_std <= threshold)
        print("First affected [case, selected_channel] pairs:")
        print(bad_pairs[:10])
        print("Selected-channel order:", FEATURES_TO_USE)

    return X[keep_mask], y[keep_mask], original_indices[keep_mask]


if X_train_5.shape[0] != len(y_train_raw):
    raise ValueError("Training tensor and label counts do not match.")
if X_test_5.shape[0] != len(y_test_raw):
    raise ValueError("Test tensor and label counts do not match.")
if not np.isfinite(X_train_5).all():
    raise ValueError("Selected training data contain NaN or infinite values.")
if not np.isfinite(X_test_5).all():
    raise ValueError("Selected test data contain NaN or infinite values.")

X_train, y_train, train_original_indices = stratified_sample(
    X_train_5, y_train_raw, TRAIN_SAMPLE_SIZE, RANDOM_SEED
)
X_test, y_test, test_original_indices = stratified_sample(
    X_test_5, y_test_raw, TEST_SAMPLE_SIZE, RANDOM_SEED
)

if DROP_LOW_VARIANCE_CASES:
    X_train, y_train, train_original_indices = remove_low_variance_cases(
        X_train,
        y_train,
        train_original_indices,
        VARIANCE_THRESHOLD,
        "Training data",
    )
    X_test, y_test, test_original_indices = remove_low_variance_cases(
        X_test,
        y_test,
        test_original_indices,
        VARIANCE_THRESHOLD,
        "Test data",
    )

print("Final training shape:", X_train.shape, y_train.shape)
print("Final test shape:    ", X_test.shape, y_test.shape)
print("\nTraining classes")
display(class_count_table(y_train))
print("Test classes")
display(class_count_table(y_test))

all_labels = np.unique(np.concatenate([y_train, y_test]))
if len(all_labels) != 2:
    raise ValueError(f"Expected binary labels, found {all_labels}")
if 1 not in all_labels:
    raise ValueError(f"Expected positive class label 1, found {all_labels}")

positive_label = 1
negative_label = int(all_labels[all_labels != positive_label][0])
print("Negative label:", negative_label)
print("Positive label:", positive_label)


## 6. Fit the QUANT transform once and cache the transformed training matrix

`QUANTTransformer` performs the deterministic QUANT interval/quantile feature
extraction. The class-weight search is then performed only on the transformed tabular
features, so the expensive time-series transformation is not repeated for every class
weight.


In [ ]:
def index_hash(indices):
    return hashlib.sha256(np.asarray(indices, dtype=np.int64).tobytes()).hexdigest()


def transformed_to_numpy(matrix):
    """Convert aeon transform output to a float32 NumPy array."""
    if hasattr(matrix, "detach"):
        matrix = matrix.detach().cpu().numpy()
    return np.asarray(matrix, dtype=np.float32)


transformer_path = OUTPUT_DIR / "quant_transformer_5_features.joblib"
train_cache_path = OUTPUT_DIR / "X_train_quant_5_features.npy"
train_cache_metadata_path = OUTPUT_DIR / "X_train_quant_5_features_metadata.json"

expected_train_cache_metadata = {
    "features": FEATURES_TO_USE,
    "selected_indices": selected_indices,
    "n_cases": int(len(y_train)),
    "case_index_hash": index_hash(train_original_indices),
    "interval_depth": QUANT_INTERVAL_DEPTH,
    "quantile_divisor": QUANT_QUANTILE_DIVISOR,
    "source_x_train": x_train_path.name,
    "source_y_train": y_train_path.name,
}

cache_is_valid = False
if (
    CACHE_TRANSFORMED_FEATURES
    and REUSE_VALID_CACHE
    and transformer_path.exists()
    and train_cache_path.exists()
    and train_cache_metadata_path.exists()
):
    with open(train_cache_metadata_path, "r") as f:
        saved_metadata = json.load(f)
    cache_is_valid = saved_metadata == expected_train_cache_metadata

if cache_is_valid:
    print("Loading the existing valid QUANT training-feature cache.")
    quant_transformer = joblib.load(transformer_path)
    X_train_quant = np.load(train_cache_path, mmap_mode="r")
else:
    print("Fitting QUANTTransformer once on the selected training tensor...")
    quant_transformer = QUANTTransformer(
        interval_depth=QUANT_INTERVAL_DEPTH,
        quantile_divisor=QUANT_QUANTILE_DIVISOR,
    )

    start = time.time()
    transformed = quant_transformer.fit_transform(X_train)
    X_train_quant = transformed_to_numpy(transformed)
    transform_train_seconds = time.time() - start
    print(f"Training transform time: {transform_train_seconds / 60:.2f} minutes")
    print("Transformed training shape:", X_train_quant.shape)

    if CACHE_TRANSFORMED_FEATURES:
        print("Saving transformed training features and transformer...")
        np.save(train_cache_path, X_train_quant)
        joblib.dump(quant_transformer, transformer_path)
        with open(train_cache_metadata_path, "w") as f:
            json.dump(expected_train_cache_metadata, f, indent=2)
        # Reopen as a memory map after saving.
        del X_train_quant
        gc.collect()
        X_train_quant = np.load(train_cache_path, mmap_mode="r")

print("X_train_quant:", X_train_quant.shape, X_train_quant.dtype)


## 7. Create the model-training and validation split


In [ ]:
all_train_indices = np.arange(len(y_train))
model_train_idx, validation_idx = train_test_split(
    all_train_indices,
    test_size=VALIDATION_SIZE,
    stratify=y_train,
    random_state=RANDOM_SEED,
)

model_train_idx = np.sort(model_train_idx)
validation_idx = np.sort(validation_idx)

y_model_train = y_train[model_train_idx]
y_validation = y_train[validation_idx]

print("Model-training cases:", len(model_train_idx))
print("Validation cases:    ", len(validation_idx))
print("\nModel-training classes")
display(class_count_table(y_model_train))
print("Validation classes")
display(class_count_table(y_validation))


## 8. Metric and probability helpers


In [ ]:
def positive_class_scores(classifier, X, positive_label=1, batch_size=20_000):
    classes = list(classifier.classes_)
    if positive_label not in classes:
        raise ValueError(f"Positive label {positive_label} not found in {classes}")
    positive_column = classes.index(positive_label)

    score_chunks = []
    for start in tqdm(range(0, len(X), batch_size), desc="Predicting probabilities"):
        stop = min(start + batch_size, len(X))
        score_chunks.append(classifier.predict_proba(X[start:stop])[:, positive_column])
    return np.concatenate(score_chunks)


def labels_from_threshold(scores, threshold, dtype=None):
    predictions = np.where(scores >= threshold, positive_label, negative_label)
    return predictions.astype(dtype) if dtype is not None else predictions


def binary_metrics(y_true, y_pred, scores=None):
    true_positive_mask = np.asarray(y_true) == positive_label
    pred_positive_mask = np.asarray(y_pred) == positive_label

    tn, fp, fn, tp = confusion_matrix(
        true_positive_mask,
        pred_positive_mask,
        labels=[False, True],
    ).ravel()

    pod = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    far = fp / (tp + fp) if (tp + fp) else np.nan
    tss = pod - fpr if np.isfinite(pod) and np.isfinite(fpr) else np.nan

    hss_denominator = ((tp + fn) * (fn + tn)) + ((tp + fp) * (fp + tn))
    hss = (
        2 * ((tp * tn) - (fp * fn)) / hss_denominator
        if hss_denominator
        else np.nan
    )

    output = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_positive": precision_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "recall_positive": recall_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "f1_positive": f1_score(
            y_true, y_pred, pos_label=positive_label, zero_division=0
        ),
        "POD_recall": pod,
        "FPR": fpr,
        "FAR": far,
        "TSS": tss,
        "HSS": hss,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
    }

    if scores is not None:
        output["roc_auc"] = roc_auc_score(true_positive_mask, scores)
        output["average_precision_pr_auc"] = average_precision_score(
            true_positive_mask, scores
        )

    return output


def build_extra_trees(class_weight):
    return ExtraTreesClassifier(
        n_estimators=N_ESTIMATORS,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )


## 9. Search class weights and probability thresholds

For each class weight, the classifier is fitted once on the transformed model-training
split. Validation probabilities are then reused across all threshold candidates.


In [ ]:
search_rows = []
weight_fit_rows = []

for weight_name, class_weight in CLASS_WEIGHT_OPTIONS:
    print("\n" + "=" * 72)
    print("Class weight:", weight_name, class_weight)

    classifier = build_extra_trees(class_weight)

    start = time.time()
    classifier.fit(X_train_quant[model_train_idx], y_model_train)
    fit_seconds = time.time() - start

    validation_scores = positive_class_scores(
        classifier,
        X_train_quant[validation_idx],
        positive_label=positive_label,
        batch_size=PREDICT_BATCH_SIZE,
    )

    for threshold in THRESHOLDS:
        validation_pred = labels_from_threshold(
            validation_scores,
            threshold,
            dtype=y_validation.dtype,
        )
        metrics = binary_metrics(
            y_validation,
            validation_pred,
            scores=validation_scores,
        )
        search_rows.append({
            "class_weight_name": weight_name,
            "class_weight": str(class_weight),
            "threshold": float(threshold),
            "classifier_fit_seconds": fit_seconds,
            **metrics,
        })

    weight_fit_rows.append({
        "class_weight_name": weight_name,
        "class_weight": str(class_weight),
        "classifier_fit_seconds": fit_seconds,
    })

    print(f"Classifier fit time: {fit_seconds / 60:.2f} minutes")
    del classifier, validation_scores
    gc.collect()

validation_search_df = pd.DataFrame(search_rows)

# Highest TSS wins. HSS and precision are deterministic tie-breakers.
ranked_search_df = validation_search_df.sort_values(
    ["TSS", "HSS", "precision_positive"],
    ascending=[False, False, False],
).reset_index(drop=True)

best_row = ranked_search_df.iloc[0]
best_weight_name = best_row["class_weight_name"]
best_threshold = float(best_row["threshold"])
best_class_weight = dict(CLASS_WEIGHT_OPTIONS)[best_weight_name]

print("\nBest validation configuration")
print("Class weight:", best_weight_name, best_class_weight)
print(f"Threshold: {best_threshold:.2f}")
print(f"Validation TSS: {best_row['TSS']:.4f}")
print(f"Validation HSS: {best_row['HSS']:.4f}")
print(f"Validation balanced accuracy: {best_row['balanced_accuracy']:.4f}")

print("\nTop 15 class-weight/threshold combinations")
display(ranked_search_df.head(15))


## 10. Compare the best threshold for each class weight


In [ ]:
best_per_weight_df = (
    validation_search_df
    .sort_values(
        ["class_weight_name", "TSS", "HSS", "precision_positive"],
        ascending=[True, False, False, False],
    )
    .groupby("class_weight_name", as_index=False)
    .first()
    .sort_values("TSS", ascending=False)
    .reset_index(drop=True)
)

summary_columns = [
    "class_weight_name",
    "class_weight",
    "threshold",
    "TSS",
    "HSS",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "FPR",
    "FAR",
    "classifier_fit_seconds",
]
display(best_per_weight_df[summary_columns])


## 11. Refit the winning Extra Trees classifier on all transformed training data


In [ ]:
final_classifier = build_extra_trees(best_class_weight)

start = time.time()
final_classifier.fit(X_train_quant, y_train)
final_fit_seconds = time.time() - start

print("Winning class weight:", best_weight_name, best_class_weight)
print(f"Frozen threshold: {best_threshold:.2f}")
print(f"Final classifier fit time: {final_fit_seconds / 60:.2f} minutes")


## 12. Transform and evaluate the held-out test set once


In [ ]:
test_cache_path = OUTPUT_DIR / "X_test_quant_5_features.npy"
test_cache_metadata_path = OUTPUT_DIR / "X_test_quant_5_features_metadata.json"

expected_test_cache_metadata = {
    "features": FEATURES_TO_USE,
    "selected_indices": selected_indices,
    "n_cases": int(len(y_test)),
    "case_index_hash": index_hash(test_original_indices),
    "interval_depth": QUANT_INTERVAL_DEPTH,
    "quantile_divisor": QUANT_QUANTILE_DIVISOR,
    "source_x_test": x_test_path.name,
    "source_y_test": y_test_path.name,
    "transformer_training_case_index_hash": index_hash(train_original_indices),
}

test_cache_is_valid = False
if (
    CACHE_TRANSFORMED_FEATURES
    and REUSE_VALID_CACHE
    and test_cache_path.exists()
    and test_cache_metadata_path.exists()
):
    with open(test_cache_metadata_path, "r") as f:
        saved_test_metadata = json.load(f)
    test_cache_is_valid = saved_test_metadata == expected_test_cache_metadata

if test_cache_is_valid:
    print("Loading the existing valid QUANT test-feature cache.")
    X_test_quant = np.load(test_cache_path, mmap_mode="r")
else:
    print("Applying the fitted QUANT transformer to the held-out test set...")
    start = time.time()
    transformed_test = quant_transformer.transform(X_test)
    X_test_quant = transformed_to_numpy(transformed_test)
    test_transform_seconds = time.time() - start
    print(f"Test transform time: {test_transform_seconds / 60:.2f} minutes")

    if CACHE_TRANSFORMED_FEATURES:
        np.save(test_cache_path, X_test_quant)
        with open(test_cache_metadata_path, "w") as f:
            json.dump(expected_test_cache_metadata, f, indent=2)
        del X_test_quant
        gc.collect()
        X_test_quant = np.load(test_cache_path, mmap_mode="r")

print("Transformed test shape:", X_test_quant.shape)

test_scores = positive_class_scores(
    final_classifier,
    X_test_quant,
    positive_label=positive_label,
    batch_size=PREDICT_BATCH_SIZE,
)

y_test_pred_default = labels_from_threshold(
    test_scores,
    DEFAULT_THRESHOLD,
    dtype=y_test.dtype,
)
y_test_pred_optimized = labels_from_threshold(
    test_scores,
    best_threshold,
    dtype=y_test.dtype,
)

metrics_default = binary_metrics(
    y_test,
    y_test_pred_default,
    scores=test_scores,
)
metrics_optimized = binary_metrics(
    y_test,
    y_test_pred_optimized,
    scores=test_scores,
)

comparison_df = pd.DataFrame([
    {
        "setting": f"winning_weight_default_threshold_{DEFAULT_THRESHOLD:.2f}",
        "class_weight_name": best_weight_name,
        "class_weight": str(best_class_weight),
        "threshold": DEFAULT_THRESHOLD,
        **metrics_default,
    },
    {
        "setting": "winning_weight_validation_optimized_threshold",
        "class_weight_name": best_weight_name,
        "class_weight": str(best_class_weight),
        "threshold": best_threshold,
        **metrics_optimized,
    },
]).set_index("setting")

compact_columns = [
    "class_weight_name",
    "threshold",
    "accuracy",
    "balanced_accuracy",
    "precision_positive",
    "recall_positive",
    "F1_positive" if "F1_positive" in comparison_df.columns else "f1_positive",
    "TSS",
    "HSS",
    "FPR",
    "FAR",
    "roc_auc",
    "average_precision_pr_auc",
    "TP",
    "TN",
    "FP",
    "FN",
]
compact_columns = [c for c in compact_columns if c in comparison_df.columns]
display(comparison_df[compact_columns])

print("\nClassification report — optimized threshold")
print(classification_report(y_test, y_test_pred_optimized, zero_division=0))

labels_for_cm = [negative_label, positive_label]
cm = confusion_matrix(y_test, y_test_pred_optimized, labels=labels_for_cm)
cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in labels_for_cm],
    columns=[f"pred_{x}" for x in labels_for_cm],
)
print("Confusion matrix — optimized threshold")
display(cm_df)


## 13. Save the search results, final metrics, predictions, transformer, and classifier


In [ ]:
def make_json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): make_json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_json_safe(v) for v in value]
    try:
        json.dumps(value)
        return value
    except TypeError:
        return str(value)


validation_search_path = OUTPUT_DIR / "class_weight_threshold_validation_search.csv"
best_per_weight_path = OUTPUT_DIR / "best_threshold_per_class_weight.csv"
test_comparison_path = OUTPUT_DIR / "test_metric_comparison.csv"
predictions_path = OUTPUT_DIR / "test_predictions.csv"
final_metrics_path = OUTPUT_DIR / "final_metrics.json"
final_classifier_path = OUTPUT_DIR / "final_extra_trees_classifier.joblib"

validation_search_df.to_csv(validation_search_path, index=False)
best_per_weight_df.to_csv(best_per_weight_path, index=False)
comparison_df.to_csv(test_comparison_path)

predictions_df = pd.DataFrame({
    "original_test_index": test_original_indices,
    "y_true": y_test,
    "positive_probability": test_scores,
    "y_pred_default_0_50": y_test_pred_default,
    "y_pred_optimized": y_test_pred_optimized,
    "optimized_threshold": best_threshold,
    "winning_class_weight": best_weight_name,
})
predictions_df.to_csv(predictions_path, index=False)

final_metrics = {
    "features": FEATURES_TO_USE,
    "selected_feature_indices": feature_index_map,
    "data_dir": str(DATA_DIR),
    "n_training_cases": int(len(y_train)),
    "n_validation_cases": int(len(validation_idx)),
    "n_test_cases": int(len(y_test)),
    "validation_size": VALIDATION_SIZE,
    "class_weight_candidates": {
        name: str(value) for name, value in CLASS_WEIGHT_OPTIONS
    },
    "threshold_candidates": THRESHOLDS.tolist(),
    "winning_class_weight_name": best_weight_name,
    "winning_class_weight": best_class_weight,
    "winning_threshold": best_threshold,
    "winning_validation_metrics": best_row.to_dict(),
    "test_metrics_default_threshold": metrics_default,
    "test_metrics_optimized_threshold": metrics_optimized,
    "final_classifier_fit_seconds": final_fit_seconds,
    "quant_transformer_parameters": quant_transformer.get_params(),
    "extra_trees_parameters": final_classifier.get_params(),
}

with open(final_metrics_path, "w") as f:
    json.dump(make_json_safe(final_metrics), f, indent=2)

# Save the fitted transformer even when transform caching is disabled.
joblib.dump(quant_transformer, transformer_path)

if SAVE_FINAL_MODEL:
    joblib.dump(final_classifier, final_classifier_path)

print("Saved validation grid:       ", validation_search_path)
print("Saved per-weight summary:    ", best_per_weight_path)
print("Saved test comparison:       ", test_comparison_path)
print("Saved predictions:           ", predictions_path)
print("Saved final metrics:         ", final_metrics_path)
print("Saved QUANT transformer:      ", transformer_path)
if SAVE_FINAL_MODEL:
    print("Saved final classifier:       ", final_classifier_path)


## Final interpretation

Use the row labeled `winning_weight_validation_optimized_threshold` in
`test_metric_comparison.csv` as the final held-out result.

The selected class weight and probability threshold were chosen only from the
training/validation data. The test tensor was evaluated after those choices were
frozen.


## Save final Model